<h1>Plot number of loaded frames at every iteration, for a set of recons</h1>

In [1]:
import sys
import os

# Add the path to the `simulated_recons/` folder to the Python path
sys.path.append(os.path.abspath("../../.."))
import livedifferences

from ptypy import io
import numpy as np
import re
import glob
import matplotlib.pyplot as plt
import copy
%matplotlib widget
plt.ion()


WARNING ptypy - Message Passaging for Python (mpi4py) not found.
    CPU-parallelization disabled.
    Install python-mpi4py via the package repositories or with `pip install --user mpi4py`


In [2]:
%matplotlib -l

Available matplotlib backends: ['tk', 'gtk', 'gtk3', 'gtk4', 'wx', 'qt4', 'qt5', 'qt6', 'qt', 'osx', 'nbagg', 'notebook', 'agg', 'svg', 'pdf', 'ps', 'inline', 'ipympl', 'widget']


In [3]:
npzfile = np.load("recon_info.npz", allow_pickle=True)
print(npzfile.files)  # prints the keys

live_info = list(npzfile['live_info'])
livesim_info = list(npzfile['livesim_info'])
offl_info = list(npzfile['offl_info'])

reclist_live = [recon for recon in live_info if recon["sample"]=="broken_gold"]
reclist_livesim = [recon for recon in livesim_info if recon["sample"]=="broken_gold"]
scannrs = [recdict['scan'] for recdict in reclist_livesim]


['live_info', 'livesim_info', 'offl_info']


In [15]:
scannrs = [recdict['scan'] for recdict in reclist_livesim]
scannrs = list(dict.fromkeys(scannrs)) # remove duplicate items in the list
scannrs
fnamedict = {}
for scan in scannrs:
    fnamedict[scan] = [recdict['rec_path'][-1] for recdict in reclist_live+reclist_livesim if recdict["scan"]==scan]
#fnamedict['000047'].append('/data/visitors/nanomax/20250057/2025021508/process/RL/LiveSimPtycho/broken_gold/000047_20/dumps/dump_scan_000020_DM_pycuda_0200.ptyr')

In [17]:
## Debug the cell below
datadict = {}

# how I want the structure: datadict['000057']['12'].keys() = 'path', 'it', 'frames'
for scan in scannrs:
    recsuffix = [re.findall(r'(?<=\d{6}_)\d{2}',fname)[0] for fname in fnamedict[scan]]
    datadict[scan] = {}
    for i, sfx in enumerate(recsuffix):
        datadict[scan][sfx] = {}
        path = fnamedict[scan][i]#.rsplit('/',2)[0]
        datadict[scan][sfx]['path'] = path
        
        fname = glob.glob(path.rsplit('/',2)[0] + '/backtrace-summary*')[0]
        with open(fname, 'r') as f:
            fpi_str = f.read()
        # Extract the numbers in the file. Every other entry in this list corresponds to nr 
        # of frames that have been loaded and to nr of iterations that have been performed.
        fpi_data_flattened = [int(s) for s in re.findall(r'\b\d+\b', fpi_str)] 
        frames = fpi_data_flattened[::2]
        it = fpi_data_flattened[1::2]
        datadict[scan][sfx]['frames'] = frames
        datadict[scan][sfx]['it'] = it
        
#        # Add manually
#        totit = int(path.rstrip('.ptyr').rsplit('_',1)[-1])
#        list(datadict['000057'].keys())
#        for key in keylist:
#            print(datadict['000057'][key]['it'][-1])
#        
#        datadict[scan][sfx]['frames'] = frames
#        datadict[scan][sfx]['it'] = it#

    

In [183]:

"""
datadict = copy.deepcopy(fnamedict)

for scan in scannrs:
    datadict[scan] = {'paths': [fname.rsplit('/',2)[0] for fname in datadict[scan]]}
    
    datadict[scan]['frames'] = {}
    datadict[scan]['it'] = {}


frames = []
it = []
for scan in scannrs:
    paths = datadict[scan]['paths']
    for k, path in enumerate(paths):
        # Load nr. of frames loaded at each iteration.
        fname = glob.glob(path + '/backtrace-summary*')[0]
        with open(fname, 'r') as f:
            fpi_str = f.read()
        # Extract the numbers in the file. Every other entry in this list corresponds to nr 
        # of frames that have been loaded and to nr of iterations that have been performed.
        fpi_data_flattened = [int(s) for s in re.findall(r'\b\d+\b', fpi_str)] 
        frames.append(fpi_data_flattened[::2])
        it.append(fpi_data_flattened[1::2])
        #print(f'scan: {scan},k: {k}, len(frames): {len(frames)}')
        
        datadict[scan]['frames'][k] = frames[k]
        datadict[scan]['it'][k] = it[k]

#print(len(datadict['000047']['it']), datadict['000047']['it'][3][:-5])

""";
""" last iterations for scan 000057_{01-12} should be:
rec_scan_000001_DM_pycuda_2201.ptyr
rec_scan_000002_DM_pycuda_2219.ptyr
rec_scan_000003_DM_pycuda_2205.ptyr
rec_scan_000004_DM_pycuda_2203.ptyr
rec_scan_000005_DM_pycuda_2224.ptyr
rec_scan_000006_DM_pycuda_2216.ptyr
rec_scan_000007_DM_pycuda_2221.ptyr
rec_scan_000008_DM_pycuda_2196.ptyr
rec_scan_000009_DM_pycuda_2227.ptyr
rec_scan_000010_DM_pycuda_2211.ptyr
rec_scan_000011_DM_pycuda_2213.ptyr
rec_scan_000012_DM_pycuda_2230.ptyr
""";
    
# ToDo: Add last data points manually since it's not written to file.
# for ol in ['5702', '5700']: 
#     # it[ol].append(it[ol][-1]+1)
#     # frames[ol].append(315
#     totit = int(realtime_fname.rstrip('.ptyr').rsplit('_',1)[-1])
#     additer = list(np.arange(it[ol][-1]+1,totit+1))
#     it[ol].extend(additer)
#     frames[ol].extend(list(np.ones(len(additer), dtype='int')*315)) # 315 is the number if frames in total




# for ol, path in zip(['5702', '5700'],[path5702, path5700]):
    
#     # Load nr. of frames loaded at each iteration.
#     fname = glob.glob(path + '/backtrace-summary*')[0]
#     with open(fname, 'r') as f:
#         fpi_str = f.read()
#     # Extract the numbers in the file. Every other entry in this list corresponds to nr 
#     # of frames that have been loaded and to nr of iterations that have been performed.
#     fpi_data_flattened = [int(s) for s in re.findall(r'\b\d+\b', fpi_str)] 
#     frames[ol] = fpi_data_flattened[::2]
#     it[ol] = fpi_data_flattened[1::2]
    
# # ToDo: Add last data points manually since it's not written to file.
# for ol in ['5702', '5700']: 
#     # it[ol].append(it[ol][-1]+1)
#     # frames[ol].append(315
#     totit = int(realtime_fname.rstrip('.ptyr').rsplit('_',1)[-1])
#     additer = list(np.arange(it[ol][-1]+1,totit+1))
#     it[ol].extend(additer)
#     frames[ol].extend(list(np.ones(len(additer), dtype='int')*315)) # 315 is the number if frames in total

In [184]:
print(len(datadict['000047']['13']['it']), fnamedict['000047'][13], datadict['000047']['13']['it'][-5:])

2560 /data/visitors/nanomax/20250057/2025021508/process/RL/LiveSimPtycho/broken_gold/000047_13/rec/rec_scan_000013_DM_pycuda_2659.ptyr [2555, 2556, 2557, 2558, 2559]


In [28]:
### To be modified ###
def plot_FPI(savefigs=False, scan='000057'):
    cmap = plt.get_cmap('Spectral')
    n_colors = 5
    # Generate n evenly spaced values between 0 and 1
    colors = [cmap(i / (n_colors - 1)) for i in range(n_colors)]
    colors[2] = [0.993248, 0.906157, 0.143936, 1.0] # changing the yellow to a less bright one.

    # Plot
    params = {'legend.fontsize': 13, # 20,
             'axes.labelsize': 17, # 25.2,
             'axes.titlesize': 17, # 25.2,
             'xtick.labelsize': 13, # 20,
             'ytick.labelsize': 13} # 20}
    plt.rcParams.update(params)
    fig, ax = plt.subplots(figsize = (9,6))
    #fig_fpi.figsize = (30,14)
    fig.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.95, wspace=0.40, hspace=0.2)
    
    
    
    recsuffix = list(datadict[scan].keys())
    
    sfx = '00'
    ax.plot(datadict[scan][sfx]['it'], datadict[scan][sfx]['frames'], 'kd-', linewidth=1.75, markersize=2 )
    recsuffix = [sfx for sfx in recsuffix if int(sfx) >= 20] # only plot livesim-recons with suffix >= 13
    recsuffix.insert(0,'00')
    for sfx in recsuffix[1:]:
            ax.plot(datadict[scan][sfx]['it'], datadict[scan][sfx]['frames'], 'd-', linewidth=1, markersize=1 )
    
    
    #for k,ol in enumerate(['5702', '5700']): 
    #    color = list(colors[k])
    #    color[-1] = 0.99
    #    ax.plot(it[ol], frames[ol], 'd-', linewidth=1, markersize=1, c=color)#(1-1/(k+1), 1-1/(k+1), 1/(k+1), 0.25))
    ax.legend(recsuffix)
    #ax.legend(['90.0%', '80.7%', '71.0%', '60.9%', '54.2%'], frameon=False)

    ax.set_xlabel('Iterations')
    ax.set_ylabel('Frames')
    fig.suptitle(scan)

    ax.set_ylim(bottom=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # fr = np.array(datadict[scan][sfx]['frames'])
    # iterticks = [np.where(fr == k)[0][-1] for k in range(50,325,25)]
    # ax.set_xticks(iterticks, minor=True)
    # ax.set_yticks([0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 315], minor=True)
    # ax.grid(which='minor')
    
    #fr = np.array(frames)
    #iterticks = [np.where(fr == k)[0][-1] for k in range(50,325,25)]
    #ax_fpi1.set_xticks(iterticks, minor=True)
    #ax_fpi1.set_yticks([0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 315], minor=True)
    #ax_fpi1.grid(which='minor')
    plt.show()
    if savefigs:
        #print(f'Saving image to /.../ptyrplots/Frames_per_iteration.png')
        fig.savefig(f'figures/Frames_per_iteration_{scan}.png', dpi=400)

    #ax.get_box_aspect
    #colors
    return fig, ax
fig0, ax0 = plot_FPI(savefigs=True, scan='000057')
#ax0.set_xlim(left=-50,right=1000)
#ax0.set_ylim(bottom=None,top=450)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [150]:
recsuffix_ = [sfx for sfx in recsuffix if int(sfx) >= 13]
recsuffix_.insert(0,'00')
len(recsuffix_), recsuffix_

(8, ['00', '13', '14', '15', '16', '17', '18', '19'])

<h2>Old</h2>

In [4]:

frames = {}
it = {}
for ol, path in zip(['5702', '5700'],[path5702, path5700]):
    
    # Load nr. of frames loaded at each iteration.
    fname = glob.glob(path + '/backtrace-summary*')[0]
    with open(fname, 'r') as f:
        fpi_str = f.read()
    # Extract the numbers in the file. Every other entry in this list corresponds to nr 
    # of frames that have been loaded and to nr of iterations that have been performed.
    fpi_data_flattened = [int(s) for s in re.findall(r'\b\d+\b', fpi_str)] 
    frames[ol] = fpi_data_flattened[::2]
    it[ol] = fpi_data_flattened[1::2]
    
# ToDo: Add last data points manually since it's not written to file.
for ol in ['5702', '5700']: 
    # it[ol].append(it[ol][-1]+1)
    # frames[ol].append(315
    totit = int(realtime_fname.rstrip('.ptyr').rsplit('_',1)[-1])
    additer = list(np.arange(it[ol][-1]+1,totit+1))
    it[ol].extend(additer)
    frames[ol].extend(list(np.ones(len(additer), dtype='int')*315)) # 315 is the number if frames in total




""" # old versoin:
path5702 = LD5702.fname2.rsplit('/',2)[0]
path5700 = LD5700.fname2.rsplit('/',2)[0]
#path27 = LD27.fname2.rsplit('/',2)[0]
#path35 = LD35.fname2.rsplit('/',2)[0]
#path40 = LD40.fname2.rsplit('/',2)[0]

frames = {}
it = {}
for ol, path in zip(['5702', '5700'],[path5702, path5700]):
    
    # Load nr. of frames loaded at each iteration.
    fname = glob.glob(path + '/backtrace-summary*')[0]
    with open(fname, 'r') as f:
        fpi_str = f.read()
    # Extract the numbers in the file. Every other entry in this list corresponds to nr 
    # of frames that have been loaded and to nr of iterations that have been performed.
    fpi_data_flattened = [int(s) for s in re.findall(r'\b\d+\b', fpi_str)] 
    frames[ol] = fpi_data_flattened[::2]
    it[ol] = fpi_data_flattened[1::2]
    
# ToDo: Add last data points manually since it's not written to file.
for ol in ['5702', '5700']: 
    # it[ol].append(it[ol][-1]+1)
    # frames[ol].append(315
    totit = int(realtime_fname.rstrip('.ptyr').rsplit('_',1)[-1])
    additer = list(np.arange(it[ol][-1]+1,totit+1))
    it[ol].extend(additer)
    frames[ol].extend(list(np.ones(len(additer), dtype='int')*315)) # 315 is the number if frames in total

"""

In [127]:
savefigs=False
cmap = plt.get_cmap('Spectral')
n_colors = 5
# Generate n evenly spaced values between 0 and 1
colors = [cmap(i / (n_colors - 1)) for i in range(n_colors)]
colors[2] = [0.993248, 0.906157, 0.143936, 1.0] # changing the yellow to a less bright one.

# Plot
params = {'legend.fontsize': 13, # 20,
         'axes.labelsize': 17, # 25.2,
         'axes.titlesize': 17, # 25.2,
         'xtick.labelsize': 13, # 20,
         'ytick.labelsize': 13} # 20}
plt.rcParams.update(params)
fig, ax = plt.subplots(figsize = (9,6))
#fig_fpi.figsize = (30,14)
fig.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.95, wspace=0.40, hspace=0.2)

for k,ol in enumerate(['5702', '5700']): 

    color = list(colors[k])
    color[-1] = 0.99
    ax.plot(it[ol], frames[ol], 'd-', linewidth=1, markersize=1, c=color)#(1-1/(k+1), 1-1/(k+1), 1/(k+1), 0.25))
ax.legend(['5702', '5700'])
#ax.legend(['90.0%', '80.7%', '71.0%', '60.9%', '54.2%'], frameon=False)

ax.set_xlabel('Iterations')
ax.set_ylabel('Frames')

ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
#fr = np.array(frames)
#iterticks = [np.where(fr == k)[0][-1] for k in range(50,325,25)]
#ax_fpi1.set_xticks(iterticks, minor=True)
#ax_fpi1.set_yticks([0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 315], minor=True)
#ax_fpi1.grid(which='minor')
plt.show()
if savefigs:
    print(f'Saving image to /.../ptyrplots/Frames_per_iteration.png')
    fig.savefig(f'Frames_per_iteration.png', dpi=400)

#ax.get_box_aspect
#colors

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

type: list indices must be integers or slices, not str

In [275]:
ax.bbox.bounds

(60.0, 69.99999999999999, 510.0, 595.0)

<h2>Other</h2>

In [71]:
rawdata_paths = [recon_dict["rawdata_paths"][0] for recon_dict in reclist_livesim if recon_dict["recfoldername"][-2:] == "01"]
rawdata_paths.sort()
dts = []
for path in rawdata_paths:
    print(path)
    dts.append(io.h5read(path, 'entry/measurement/dt')['entry/measurement/dt'])


/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000047.h5
/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000049.h5
/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000051.h5
/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000053.h5
/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000055.h5
/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000057.h5


In [72]:
ddts = []
for dt_arr in dts:
    ddts.append(dt_arr[1::2] - dt_arr[::2])  # take the difference between every other timestamp


In [87]:
print("scan   min     max     mean    median  std")
for i, ddt in enumerate(ddts):
    print(scannrs[i], f"{np.min(ddt):.04f}, {np.max(ddt):.04f}, {np.mean(ddt):.04f}, {np.median(ddt):.04f}, {np.std(ddt):.04f}")

    """From relay_output.txt:
scan     min     max     mean    median  std
000047:  0.7135, 0.8445, 0.7602, 0.7548, 0.0238
000049:  0.7132, 0.8445, 0.7596, 0.7546, 0.0236
000051:  0.7136, 0.8555, 0.7589, 0.7545, 0.0238
000053:  0.7140, 0.8652, 0.7613, 0.7549, 0.0247
000055:  0.7139, 0.8649, 0.7610, 0.7546, 0.0257
000057:  0.7138, 1.0055, 0.7600, 0.7545, 0.0264
""";
    scan   min     max     mean    median  std
000047 0.7060, 0.8360, 0.7585, 0.7551, 0.0238
000049 0.7150, 0.8540, 0.7592, 0.7560, 0.0232
000051 0.7129, 0.8740, 0.7591, 0.7560, 0.0235
000053 0.7159, 0.8609, 0.7612, 0.7594, 0.0243
000055 0.7139, 0.8771, 0.7601, 0.7561, 0.0255
000057 0.7119, 1.0030, 0.7599, 0.7560, 0.0266

scan   min     max     mean    median  std
000047 0.7060, 0.8360, 0.7585, 0.7551, 0.0238
000049 0.7150, 0.8540, 0.7592, 0.7560, 0.0232
000051 0.7129, 0.8740, 0.7591, 0.7560, 0.0235
000053 0.7159, 0.8609, 0.7612, 0.7594, 0.0243
000055 0.7139, 0.8771, 0.7601, 0.7561, 0.0255
000057 0.7119, 1.0030, 0.7599, 0.7560, 0.0266


In [162]:
np.mean(np.array([0.7585, 0.7592, 0.7591, 0.7612, 0.7601, 0.7599]))

0.7596666666666666

In [105]:
np.set_printoptions(linewidth=120, precision=2, floatmode='fixed')  # default: linewidth=75, precision=8, floatmode='maxprec_equal'

for i, ddt in enumerate(ddts):
    print(scannrs[i], ddt[:15])

000047 [0.75 0.78 0.79 0.74 0.74 0.81 0.74 0.79 0.81 0.76 0.74 0.76 0.80 0.74 0.73]
000049 [0.77 0.73 0.74 0.78 0.74 0.77 0.78 0.75 0.74 0.79 0.74 0.75 0.74 0.77 0.77]
000051 [0.78 0.80 0.78 0.75 0.72 0.77 0.79 0.73 0.74 0.78 0.76 0.78 0.78 0.75 0.77]
000053 [0.83 0.78 0.74 0.74 0.74 0.75 0.79 0.77 0.77 0.76 0.73 0.80 0.78 0.74 0.75]
000055 [0.82 0.76 0.74 0.77 0.78 0.75 0.73 0.74 0.86 0.74 0.77 0.75 0.80 0.75 0.79]
000057 [0.84 0.78 0.73 0.75 0.73 0.74 0.76 0.73 0.85 0.75 0.72 0.74 0.81 0.73 0.77]


In [126]:
np.mean(np.array([0.7602, 0.7596, 0.7589, 0.7613, 0.7610, 0.7600])), np.median(np.array([0.7602, 0.7596, 0.7589, 0.7613, 0.7610, 0.7600]))

(0.7601666666666667, 0.7601)

In [124]:
#fig=plt.figure()
for i, ddt in enumerate(ddts):
    fig=plt.figure(figsize=(12,4))
    pl=plt.plot(ddt)
    pl[0].axes.set_ylim(bottom=0.705,top=1.004)
    plt.legend([scannrs[i]])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
self.t0